# CDC-8 — CDC schema evolution

**Break → Detect → Fix → Prove.** You run `ALTER TABLE` on the **source** Postgres table. Postgres
logical decoding does **not** emit a DDL event — there is no "schema changed" message on the topic.
Instead, the **next change event for a row touched after the `ALTER` simply carries the new shape**:
an added column appears in that event's `after` object, with **no connector restart** and **no
signal**. Additive (`ADD COLUMN`) flows through transparently; renames / drops / type-narrowing can
desync a strict downstream consumer and may need an ad-hoc snapshot.

**What we prove (deterministic):** a post-`ALTER` event's `after` keys include the new `discount`
column while a pre-`ALTER` event's keys do not — the envelope evolved on its own.
**What we describe (honesty):** (a) no explicit DDL event flows for Postgres — the change is *observed*
via the payload (and Debezium's internal, WAL-reconstructed schema), not announced; (b) **breaking**
changes (DROP/RENAME/narrow) can break downstream and may need an **`execute-snapshot` signal** or a
connector recreate — too consumer-specific to stage in one tiny run; (c) the downstream fix is to
**evolve the sink** — for Iceberg an additive `ALTER TABLE … ADD COLUMN`, which is metadata-only and
tolerates the new column (**LAK-6**, field-id tracking).

**Prerequisite:** `make cdc-up`. **Laptop-safe:** one ≤20-row table, one connector, bounded topic
reads, short bounded `sleep`s, full teardown at start and end. Isolation: table `cdc8_orders`,
connector `cdc8-orders`.

In [ ]:
from common import cdc_helpers as cdc
import json, time

TABLE = "cdc8_orders"
CONN  = "cdc8-orders"
TOPIC = cdc.topic_name(TABLE)

print("Connect REST up:", cdc.connect_up(timeout=40))
cdc.teardown(CONN, TABLE)              # clean slate (idempotent re-run): connector, slot, table, topic
cdc.seed_orders(TABLE, n=20)           # public.cdc8_orders with the v1 shape, 20 rows
print("seeded public.%s; Debezium will publish to topic: %s" % (TABLE, TOPIC))

## 1. Seed + register — the v1 envelope

`seed_orders` creates `public.cdc8_orders` with `(id, customer, amount, status, updated)` and 20 rows.
We register a standard Debezium Postgres connector (`common.cdc_helpers.debezium_pg_config` fills the
defaults: `pgoutput`, JSON converter, `snapshot.mode=initial`, `decimal.handling.mode=double`). On
first start it snapshots the 20 rows as `r` (read) events — that's our **v1 envelope** to baseline.

In [ ]:
cfg = cdc.debezium_pg_config(CONN, TABLE, snapshot_mode="initial")
reg = cdc.register_connector(CONN, cfg)
print("register -> HTTP", reg["status"])                  # 201 created
print("state    ->", cdc.wait_for_connector(CONN, timeout=60))   # RUNNING

## 2. Break it (step A) — capture a baseline change event and print `sorted(after.keys())`

Read the topic and pull a snapshot/`r` event. Its `after` object is the **v1 shape**. We record the
sorted key-set so we can diff it across the upcoming `ALTER`.

In [ ]:
ev = cdc.read_cdc_events(TOPIC)
print("op counts after snapshot:", cdc.op_counts(ev))     # {'r': 20}

baseline = next(e for e in ev if e["after"] is not None)  # first event carrying an `after`
before_keys = sorted(baseline["after"].keys())
print("\nbaseline after keys (pre-ALTER):", before_keys)
print("one baseline `after`:", baseline["after"])

## 3. Break it (step B) — `ALTER TABLE` upstream; note: **no DDL event flows**

We add a column on the **source** database. Postgres logical decoding emits **no** DDL event, so the
topic gets **nothing** from this statement on its own — the connector stays `RUNNING` and the new
column is invisible until a row actually changes. We confirm both facts.

In [ ]:
cdc.pg_exec("ALTER TABLE public.%s ADD COLUMN discount NUMERIC(10,2) DEFAULT 0" % TABLE)
print("ALTER applied on source: added discount NUMERIC(10,2) DEFAULT 0")

# the ALTER alone produces no new topic event (no DDL event in Postgres logical decoding)
time.sleep(3)
ev_after_ddl = cdc.read_cdc_events(TOPIC, max_ms=6000)
print("op counts right after the ALTER (no DML yet):", cdc.op_counts(ev_after_ddl))  # still {'r': 20}
print("connector still:", cdc.connector_status(CONN)["connector"]["state"])          # RUNNING

## 4. Break it (step C) — touch a row with the new column, then read again

Now we `INSERT` a new row that sets `discount` and `UPDATE` an existing row's `discount`. These are
the first row-changes *after* the schema change, so their change events carry the **evolved** shape.
A short `sleep` covers the WAL decode lag.

In [ ]:
cdc.pg_exec(
    "INSERT INTO public.%s (id, customer, amount, status, discount) "
    "VALUES (100, 'evolved', 50.00, 'NEW', 5.00)" % TABLE)
cdc.pg_exec("UPDATE public.%s SET discount = 2.50 WHERE id = 1" % TABLE)
time.sleep(5)                                             # streaming decode lag

ev2 = cdc.read_cdc_events(TOPIC)
print("op counts now:", cdc.op_counts(ev2))               # {'r': 20, 'c': 1, 'u': 1}

## 5. Detect it — diff the `after` key-sets across the `ALTER`

Pull a post-`ALTER` event (the `c` insert, or the `u` update) and print its `sorted(after.keys())`
next to the baseline. The set difference is the whole diagnosis.

In [ ]:
post = next(e for e in ev2 if e["op"] in ("c", "u") and e["after"] is not None)
after_keys = sorted(post["after"].keys())

print("baseline after keys (pre-ALTER) :", before_keys)
print("post-ALTER  after keys          :", after_keys)
print("new keys (after - before)       :", set(after_keys) - set(before_keys))   # {'discount'}
print("\npost-ALTER `after`:", post["after"])
print("discount value on the evolved event:", post["after"].get("discount"))

## 6. Diagnose

- **Logical decoding carries data, not DDL.** `pgoutput` decodes **row changes** from the WAL; an
  `ALTER TABLE` changes the catalog but is **not** a logical-decoding message. Debezium learns the new
  shape from the **relation metadata** on the next change for that table — so `discount` surfaces
  **only** once a row is inserted/updated after the `ALTER`, embedded in its `after`. (MySQL/Debezium
  keeps a **schema-history topic** and parses DDL; the **Postgres** connector does **not** — no DDL
  events. Know which connector you run.)
- **Additive is backward-compatible, so it flows transparently.** Old events lacked the field; new
  events have it. A consumer that reads by name and tolerates absent fields absorbs it with no change.
- **Renames / drops / narrowing are *not* backward-compatible.** A **drop** makes the field vanish from
  later events; a **rename** looks like *drop old + add new* on the wire (the exact positional break
  LAK-6 shows for Parquet); a **narrow** can make values unparseable downstream. All ride the **same
  silent channel** (still no DDL event), so the breakage shows up only at a strict consumer — late and
  far from the cause.
- **The remedy for a breaking change is an explicit re-sync** — an **ad-hoc / incremental snapshot**
  via Debezium's `execute-snapshot` **signal** (a row in a signal table), or a connector recreate, to
  repopulate downstream under the new schema. We *describe* this; a real downstream break is
  consumer-specific and not deterministic in one tiny run.

## 7. Fix it / guidance — evolve the sink additively, by field-id (ties to LAK-6)

The connector did its job; the work is making the **downstream** tolerate evolution. When the CDC
stream gains `discount`, the Iceberg mirror (the CDC-7 upsert sink) just needs the column — an
**additive** `ALTER TABLE … ADD COLUMN`, which in Iceberg is **metadata-only** (new **field-id**, no
data-file rewrite; old rows read back `NULL`). This is exactly **LAK-6**. The cell below shows the
DDL you would run on the sink — guarded so the notebook stays self-contained and Connect-safe even if
no sink table exists here (the CDC→Iceberg pipeline is CDC-7).

In [ ]:
SINK = "iceberg_catalog.default.cdc8_orders_sink"
ddl  = "ALTER TABLE %s ADD COLUMN discount DECIMAL(10,2)" % SINK
print("Downstream (Iceberg sink) evolution — additive, metadata-only, by field-id (LAK-6):")
print("  ", ddl)
print("\nEffect: new column gets a stable field-id; data_files unchanged (no rewrite);")
print("        old rows read NULL for discount; CDC-7's MERGE then lands the new value.")

# Optional: actually evolve a sink table over Spark Connect, IF the CDC-7 pipeline created one.
# Guarded so this module needs no Spark and never fails when the sink is absent.
try:
    from common.spark_session import get_spark
    spark = get_spark()
    if spark.catalog.tableExists(SINK):
        spark.sql(ddl)
        print("\napplied to existing sink; columns now:", spark.table(SINK).columns)
    else:
        print("\n(sink %s not present — build it in CDC-7; DDL shown above is the fix)" % SINK)
except Exception as e:
    print("\n(skipped live sink evolve — Spark/sink not available here:",
          str(e).splitlines()[0][:120], ")")

## 8. Prove it

The proof is the **before/after `after`-key sets**: `discount` is present in events emitted **after**
the upstream `ADD COLUMN` and absent from events emitted **before** it — and it arrived with **no DDL
event** on the topic and **no connector restart**. Schema evolution in Postgres CDC is **observed**,
not **announced**.

In [ ]:
new_keys = set(after_keys) - set(before_keys)
state = cdc.connector_status(CONN)["connector"]["state"]

print("baseline after keys (pre-ALTER) :", before_keys)
print("post-ALTER  after keys          :", after_keys)
print("new keys (after - before)       :", new_keys)
print("connector state throughout      :", state)
print("DDL event observed on topic     : none (Postgres logical decoding emits no DDL)")
print()
print("PROVED: envelope gained 'discount' via the payload, no restart ->",
      new_keys == {"discount"} and state == "RUNNING")

## 9. Takeaways & "in real production…"

- **Postgres CDC has no DDL events — schema changes ride the payload.** Detect via a **diff of event
  keys over time**, not a DDL feed. (MySQL/Debezium *does* keep a schema-history topic.)
- **Additive is safe; breaking is silent until it isn't.** `ADD COLUMN` flows to tolerant consumers;
  DROP/RENAME/narrow flow through the same channel and surface as a **downstream** error — late and far
  from the cause. Make additive the default; gate breaking changes behind a migration.
- **Evolve the sink additively, by field-id (LAK-6).** Mirror an upstream add with Iceberg
  `ALTER TABLE … ADD COLUMN`: metadata-only, no rewrite, old rows read `NULL`.
- **Breaking changes need a plan + a re-snapshot.** Coordinate consumers, then use an **ad-hoc /
  incremental snapshot** (`execute-snapshot` signal) — or recreate the connector — to repopulate
  downstream under the new schema.
- **Contracts/registry turn desync into CI failures.** At scale, enforce compatibility (registry rules
  or data contracts) so an incompatible change is rejected at publish time. Same lesson as dbt's
  `on_schema_change` (**DBT-5**), one layer up.

## 10. Teardown

In [ ]:
cdc.teardown(CONN, TABLE)     # delete connector, drop slot, drop table, delete topic
print("slots now:", cdc.list_slots())